# Kaggle — 11 Aday Ham Log İncelemesi v2 (No-Inference)

Bu sürüm, ilk notebook'taki arşiv yolu eşleştirme hatasını giderir. ZIP içindeki **gerçek tüm dosya yollarını** envanterler; bildirilen artifact yollarını normalize ederek ve yalnızca benzersiz güvenli suffix eşleşmesiyle arar. Ardından aday klasörüne ait log/metin/JSON dosyalarını ayrıca tarar.

**Güvenlik sınırı:** Model yükleme, çıkarım, eğitim, kaynak kodu çalıştırma, patch uygulama veya aday kabul kararı yoktur. Arşivdeki metinler yalnızca okunur. Otomatik kategori yalnızca triyajdır.

**Kaggle ayarları:** Internet = On, GPU = Off. Kaggle Secrets içinde `GITHUB_TOKEN` adlı, depoda Contents read/write yetkili token bulunmalı.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, json, csv, hashlib, zipfile, io, re, base64, requests

REPO = 'MahmutVezir/dijitalikiz'
BRANCH = 'research/2026-11-16-followup-evidence-validation'
BASE_PATH = 'research_candidates/evidence_abstention/kaggle_runs/2026-11-17_b4b8b536e82e'
ZIP_NAME = 'kaggle_no_inference_candidate_evidence.zip'
SUMMARY_NAME = 'kaggle_candidate_job_summary.json'
EXPECTED_ZIP_SHA256 = 'b4b8b536e82e7ac93e94bbb51d63f71185a2b7ab2515178480189f8cf783b777'
EXPECTED_SUMMARY_SHA256 = 'd0bba8618d4af349ae20348ba0a8b046fbfe0d1cc2e81e777ab3bdb27bfd564a'
OUT = Path('/kaggle/working/no_inference_log_review')
OUT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
print('Run ID:', RUN_ID)
print('Output:', OUT)


In [ ]:
# GitHub secret'i güvenli biçimde al; token hiçbir çıktıya yazdırılmaz.
try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception as exc:
    raise RuntimeError('Kaggle Secrets içinde GITHUB_TOKEN bulunamadı veya okunamadı.') from exc
if not GITHUB_TOKEN:
    raise RuntimeError('GITHUB_TOKEN boş.')
HEADERS = {'Authorization': f'Bearer {GITHUB_TOKEN}', 'Accept': 'application/vnd.github+json', 'X-GitHub-Api-Version': '2022-11-28'}

def download_repo_file(filename, expected_sha):
    url = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/{BASE_PATH}/{filename}'
    response = requests.get(url, timeout=180)
    if response.status_code != 200:
        api_url = f'https://api.github.com/repos/{REPO}/contents/{BASE_PATH}/{filename}?ref={BRANCH}'
        response = requests.get(api_url, headers={**HEADERS, 'Accept':'application/vnd.github.raw'}, timeout=180)
    response.raise_for_status()
    data = response.content
    actual = hashlib.sha256(data).hexdigest()
    if actual != expected_sha:
        raise ValueError(f'{filename} SHA-256 uyuşmuyor: {actual}')
    print(f'{filename}: {len(data):,} bytes | SHA-256 doğrulandı')
    return data

summary_bytes = download_repo_file(SUMMARY_NAME, EXPECTED_SUMMARY_SHA256)
zip_bytes = download_repo_file(ZIP_NAME, EXPECTED_ZIP_SHA256)
summary = json.loads(summary_bytes.decode('utf-8'))
assert summary.get('model_loaded') is False
assert summary.get('model_inference_performed') is False
assert summary.get('training_performed') is False
print('Summary jobs:', summary.get('completed_job_count'), '/', summary.get('expected_job_count'))
print('Model/inference/training flags: all false')


In [ ]:
# ZIP CRC bütünlüğü + dosya envanteri. İçeriği çalıştırmaz; yalnızca okur.
zf = zipfile.ZipFile(io.BytesIO(zip_bytes))
bad_member = zf.testzip()
if bad_member is not None:
    raise RuntimeError(f'ZIP CRC hatası: {bad_member}')
members = [m for m in zf.infolist() if not m.is_dir()]
print('ZIP integrity: PASS')
print('Archive files:', len(members))
archive_names = {m.filename for m in members}
print('Top-level folders:', sorted({n.split('/')[0] for n in archive_names if '/' in n}))


In [ ]:
# Başarısız adayları al; arşiv envanteri ve güvenli yol eşleştirmesi.
failed_jobs = [j for j in summary.get('jobs', []) if j.get('outcome') == 'failure']
assert len(failed_jobs) == 11, f'Beklenen 11 başarısız aday job; bulunan: {len(failed_jobs)}'

def norm_path(p):
    p = str(p or '').replace('\\', '/').strip()
    while p.startswith('./'):
        p = p[2:]
    return '/'.join(part for part in p.split('/') if part and part != '.')

archive_paths = sorted(norm_path(m.filename) for m in members)
archive_set = set(archive_paths)
archive_inventory_path = OUT / 'archive_member_inventory.csv'
with archive_inventory_path.open('w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(['archive_path', 'uncompressed_bytes', 'suffix'])
    for m in members:
        p = norm_path(m.filename)
        w.writerow([p, m.file_size, Path(p).suffix.lower()])

def resolve_declared_path(declared):
    target = norm_path(declared)
    if target in archive_set:
        return target, 'exact_normalized'
    suffix_matches = [p for p in archive_paths if p.endswith('/' + target)]
    if len(suffix_matches) == 1:
        return suffix_matches[0], 'unique_suffix'
    if len(suffix_matches) > 1:
        return None, 'ambiguous_suffix'
    return None, 'not_found'

def candidate_scope_paths(job_name):
    slug = norm_path(job_name).lower()
    # Güvenli aday kapsamı: tam job slug'ı yol bileşeni olarak veya benzersiz
    # klasör adı öneki olarak bulunmalı; yalnızca dosya adı benzerliği yeterli değil.
    scoped = []
    for p in archive_paths:
        parts = p.lower().split('/')
        if slug in parts or any(part.startswith(slug + '/') for part in parts):
            scoped.append(p)
        elif any(part == slug for part in parts[:-1]):
            scoped.append(p)
    return sorted(set(scoped))

def safe_read(member_name, limit=2_000_000):
    try:
        info = zf.getinfo(member_name)
        if info.file_size > limit:
            return f'[SKIPPED_SIZE_LIMIT:{info.file_size}]'
        return zf.read(member_name).decode('utf-8', errors='replace')
    except KeyError:
        return '[ARCHIVE_MEMBER_MISSING]'

def classify_evidence(job, texts):
    blob = '\n'.join(texts).lower()
    rules = [
        ('network_or_remote_access', ['could not resolve host', 'connection timed out', 'connection reset', 'network is unreachable', 'failed to connect', 'temporary failure in name resolution']),
        ('git_auth_or_repository_access', ['permission denied (publickey)', 'authentication failed', 'repository not found', 'could not read from remote repository', 'http 401', 'http 403']),
        ('dependency_or_environment', ['no matching distribution found', 'modulenotfounderror', 'no module named', 'could not build wheels', 'subprocess-exited-with-error', 'command not found']),
        ('patch_or_source_mismatch', ['patch failed', 'hunk failed', 'does not apply', 'patch does not apply', 'failed to apply']),
        ('test_assertion_or_reproduction_failure', ['assertionerror', 'failed (', '= fail', ' test failed', 'tests failed']),
    ]
    for category, needles in rules:
        hits = [needle for needle in needles if needle in blob]
        if hits:
            return category, hits
    step_text = '; '.join(job.get('failed_steps') or []).lower()
    if any(x in step_text for x in ['clone', 'authenticate', 'source']):
        return 'source_checkout_failure_cause_unresolved', []
    if any(x in step_text for x in ['patch', 'apply']):
        return 'patch_step_failure_cause_unresolved', []
    if any(x in step_text for x in ['test', 'selector']):
        return 'test_step_failure_cause_unresolved', []
    return 'unclassified_inspect_evidence', []

records = []
for job in failed_jobs:
    job_name = job.get('job', 'unknown-job')
    declared = []
    for artifact in job.get('artifacts', []):
        declared.extend(artifact.get('files', []))
    resolved, unresolved = [], []
    for path in declared:
        match, method = resolve_declared_path(path)
        item = {'declared_path': path, 'resolved_path': match, 'method': method}
        (resolved if match else unresolved).append(item)

    scope_paths = candidate_scope_paths(job_name)
    # Include resolved artifacts plus candidate-scoped textual evidence.
    selected_paths = set(x['resolved_path'] for x in resolved)
    selected_paths.update(p for p in scope_paths if Path(p).suffix.lower() in {
        '.log', '.txt', '.json', '.yaml', '.yml', '.patch', '.diff', '.out', '.err'
    })
    selected_paths = sorted(selected_paths)
    readable = []
    for p in selected_paths:
        if Path(p).suffix.lower() in {'.log', '.txt', '.json', '.yaml', '.yml', '.patch', '.diff', '.out', '.err'}:
            readable.append((p, safe_read(p)))
    category, trigger_terms = classify_evidence(job, [v for _, v in readable])
    excerpts = []
    for p, content in readable:
        lines = content.splitlines()
        hits = [i for i, line in enumerate(lines) if re.search(
            r'(?i)(error|exception|traceback|fatal|failed|not found|denied|timed out|could not|assertion)', line)]
        for i in hits[:30]:
            lo, hi = max(0, i-3), min(len(lines), i+4)
            excerpts.append({'file': p, 'line': i+1, 'context': '\n'.join(lines[lo:hi])[:2400]})
    records.append({
        'job': job_name,
        'outcome': job.get('outcome'),
        'failed_steps': job.get('failed_steps') or [],
        'declared_artifact_count': len(declared),
        'resolved_artifact_count': len(resolved),
        'unresolved_artifact_count': len(unresolved),
        'path_resolution': resolved + unresolved,
        'candidate_scoped_archive_file_count': len(scope_paths),
        'selected_evidence_files': selected_paths,
        'readable_log_text_files': len(readable),
        'evidence_category': category,
        'category_trigger_terms': trigger_terms,
        'log_excerpt_candidates': excerpts[:100],
        'classification_status': 'evidence_supported_triage' if trigger_terms else 'unresolved_no_specific_log_signature'
    })

print('ZIP members inventoried:', len(members))
print('Failed candidate jobs:', len(records))
for r in records:
    print(f"{r['job']}: {r['evidence_category']} | declared/resolved={r['declared_artifact_count']}/{r['resolved_artifact_count']} | scoped={r['candidate_scoped_archive_file_count']} | text={r['readable_log_text_files']} | excerpts={len(r['log_excerpt_candidates'])}")


In [ ]:
# Raporları üret. Ham log kanıtı yoksa kategori belirsiz kalır.
report = {
    'schema_version': '2.0',
    'run_id_utc': RUN_ID,
    'repository': REPO,
    'source_branch': BRANCH,
    'source_archive_path': f'{BASE_PATH}/{ZIP_NAME}',
    'source_archive_sha256': EXPECTED_ZIP_SHA256,
    'source_summary_sha256': EXPECTED_SUMMARY_SHA256,
    'zip_integrity': 'pass',
    'archive_member_count': len(members),
    'expected_failed_candidate_jobs': 11,
    'observed_failed_candidate_jobs': len(records),
    'model_loaded': False,
    'model_inference_performed': False,
    'training_performed': False,
    'source_code_executed': False,
    'patches_applied': False,
    'acceptance_decisions_made': False,
    'interpretation': 'Path resolution and evidence extraction are mechanical triage only. Categories require explicit log signatures; unresolved cases remain unresolved. A failed job does not establish candidate invalidity.',
    'candidate_reviews': records
}
json_path = OUT / 'candidate_log_review_v2.json'
json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

csv_path = OUT / 'candidate_failure_inventory_v2.csv'
with csv_path.open('w', newline='', encoding='utf-8-sig') as f:
    fields = ['job','failed_steps','evidence_category','classification_status','declared_artifact_count','resolved_artifact_count','unresolved_artifact_count','candidate_scoped_archive_file_count','readable_log_text_files','log_excerpt_count']
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    for r in records:
        w.writerow({
            'job': r['job'], 'failed_steps': json.dumps(r['failed_steps'], ensure_ascii=False),
            'evidence_category': r['evidence_category'], 'classification_status': r['classification_status'],
            'declared_artifact_count': r['declared_artifact_count'], 'resolved_artifact_count': r['resolved_artifact_count'],
            'unresolved_artifact_count': r['unresolved_artifact_count'],
            'candidate_scoped_archive_file_count': r['candidate_scoped_archive_file_count'],
            'readable_log_text_files': r['readable_log_text_files'], 'log_excerpt_count': len(r['log_excerpt_candidates'])
        })

md = ['# 11 Aday Ham Log İncelemesi v2', '', f'- Run ID UTC: `{RUN_ID}`',
      f'- ZIP SHA-256: `{EXPECTED_ZIP_SHA256}`', f'- Arşiv dosya sayısı: {len(members)}',
      '- ZIP CRC: **PASS**', '- Model çıkarımı/eğitim: **YOK**', '- Kaynak kodu çalıştırma/patch uygulama: **YOK**',
      '- Otomatik kabul kararı: **YOK**', '',
      '> Bu rapor yalnızca log imzalarına dayalı teknik triyajdır. Kanıt yoksa neden belirsiz bırakılır.', '',
      '| Aday | Kanıt kategorisi | Artifact eşleşmesi | Aday kapsam dosyası | Log/metin |',
      '|---|---|---:|---:|---:|']
for r in records:
    md.append(f"| `{r['job']}` | `{r['evidence_category']}` | {r['resolved_artifact_count']}/{r['declared_artifact_count']} | {r['candidate_scoped_archive_file_count']} | {r['readable_log_text_files']} |")
for r in records:
    md += ['', f"## {r['job']}", '', f"- Başarısız adımlar: {', '.join(r['failed_steps']) or '—'}",
            f"- Kategori: `{r['evidence_category']}`", f"- Sınıflandırma durumu: `{r['classification_status']}`",
            f"- Artifact yolu eşleşmesi: {r['resolved_artifact_count']}/{r['declared_artifact_count']}",
            f"- Aday kapsamındaki arşiv dosyası: {r['candidate_scoped_archive_file_count']}", '']
    if r['category_trigger_terms']:
        md.append('Tetikleyici log ifadeleri: ' + ', '.join(r['category_trigger_terms']))
    for e in r['log_excerpt_candidates']:
        md += ['', f"**{e['file']}:{e['line']}**", '```text', e['context'], '```']
md_path = OUT / 'candidate_log_review_v2.md'
md_path.write_text('\n'.join(md) + '\n', encoding='utf-8')


In [ ]:
# Dört raporu GitHub'a yükle; token çıktıya veya dosyaya yazılmaz.
api_base = f'https://api.github.com/repos/{REPO}/contents'
def upload_text_file(local_path, repo_path):
    raw = local_path.read_bytes()
    url = f'{api_base}/{repo_path}'
    existing = requests.get(url, headers=HEADERS, params={'ref': BRANCH}, timeout=60)
    sha = existing.json().get('sha') if existing.status_code == 200 else None
    if existing.status_code not in (200,404):
        raise RuntimeError(f'GitHub mevcut dosya kontrolü başarısız: HTTP {existing.status_code}')
    payload = {'message': f'Add Kaggle candidate log review v2 {RUN_ID}',
               'content': base64.b64encode(raw).decode('ascii'), 'branch': BRANCH}
    if sha:
        payload['sha'] = sha
    response = requests.put(url, headers=HEADERS, json=payload, timeout=120)
    if response.status_code not in (200,201):
        raise RuntimeError(f'GitHub upload failed HTTP {response.status_code}: {response.text[:1000]}')
    return response.json()['content']['html_url'], response.json()['commit']['sha']

dest_dir = f'research_candidates/evidence_abstention/kaggle_runs/log_review_v2_{RUN_ID}'
outputs = [
    (json_path, 'candidate_log_review_v2.json'),
    (csv_path, 'candidate_failure_inventory_v2.csv'),
    (md_path, 'candidate_log_review_v2.md'),
    (archive_inventory_path, 'archive_member_inventory.csv')
]
uploaded = []
for local, name in outputs:
    url, commit = upload_text_file(local, f'{dest_dir}/{name}')
    uploaded.append({'file': name, 'url': url, 'commit': commit,
                     'bytes': local.stat().st_size, 'sha256': hashlib.sha256(local.read_bytes()).hexdigest()})
print(json.dumps({'upload_status':'SUCCESS','run_id':RUN_ID,'files':uploaded}, ensure_ascii=False, indent=2))


## Çalıştırma

1. Notebook'u Kaggle'a aktar.
2. Internet = On, GPU = Off.
3. Kaggle Secrets içinde `GITHUB_TOKEN` bulunduğunu doğrula.
4. **Run All** çalıştır.
5. Son hücrede `upload_status: SUCCESS` görünmeli.

Dört çıktı aynı `log_review_v2_<RUN_ID>` klasörüne yüklenir. Son hücredeki bağlantıları buraya gönder. Ham loglardan kanıtlanmayan hata nedenlerini belirsiz bırakacağım.